In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("t20i_info.csv")

In [9]:
df.head()

,Unnamed: 0,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,0,2,Australia,Sri Lanka,0.1,0,0,NaN,Melbourne Cricket Ground
1,1,2,Australia,Sri Lanka,0.2,0,0,NaN,Melbourne Cricket Ground
2,2,2,Australia,Sri Lanka,0.3,1,0,NaN,Melbourne Cricket Ground
3,3,2,Australia,Sri Lanka,0.4,2,0,NaN,Melbourne Cricket Ground
4,4,2,Australia,Sri Lanka,0.5,0,0,NaN,Melbourne Cricket Ground


In [4]:
df.isnull().sum()

Unnamed: 0             0
match_id               0
batting_team           0
bowling_team           0
ball                   0
runs                   0
player_dismissed       0
city                2060
venue                  1
dtype: int64

In [10]:
df.shape

(15230, 9)

In [6]:
# Cricbuzz, Cricket Line Guru, Tv Screen -> Projected Score / Predicted Score

batting team,
bowling team,
city,
current score,
balls left,
wicket left,
current run rate,
last five/six->powerplay,

In [7]:
df[df['city'].isnull()]['venue'][0].split(" ")[0]

'Melbourne'

In [11]:
df['city'] = df['city'].fillna(df['venue'].apply(lambda x: x.split(' ')[0]))

AttributeError: 'float' object has no attribute 'split'

In [ ]:
df.tail()

In [ ]:
df.isnull().sum()

In [ ]:
df['city'].value_counts()

In [ ]:
eligible_cites = df['city'].value_counts()[df['city'].value_counts() > 600].index.tolist()

In [ ]:
eligible_cites

In [ ]:
df = df[df['city'].isin(eligible_cites)]

In [ ]:
df.shape

In [ ]:
df

In [ ]:
df['current_score'] = df.groupby('match_id').cumsum()['runs']

In [ ]:
df.head()

In [ ]:
df['over'] = df['ball'].apply(lambda x : str(x).split(".")[0])
df['ball_no'] = df['ball'].apply(lambda x : str(x).split(".")[1])

In [ ]:
df.head()

In [ ]:
df['ball_bowled'] = (df['over'].astype(int)*6 + df['ball_no'].astype(int))

In [ ]:
df.tail()

In [ ]:
df['balls_left'] = 120 - df['ball_bowled']

In [ ]:
df.tail()

In [ ]:
df['balls_left'] = df['balls_left'].apply(lambda x:0 if x<0 else x)

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df['player_dismissed'] = df['player_dismissed'].apply(lambda x:1 if x!='0' else '0')

In [ ]:
df

In [ ]:
df['player_dismissed'] = df['player_dismissed'].astype(int)

In [ ]:
df['player_dismissed'] = df.groupby('match_id').cumsum()['player_dismissed']

In [ ]:
df['wicket_left'] = 10 - df['player_dismissed']

In [ ]:
df.tail()

In [ ]:
df['current_run_rate'] = (df['current_score']*6) / df['ball_bowled']

In [ ]:
df.tail()

In [ ]:
groups = df.groupby('match_id')
# 5 over = 30 ball
match_id = df['match_id'].unique()
last_five=[]
for id in match_id:
    last_five.extend(groups.get_group(id).rolling(window = 30).sum()['runs'].values.tolist())

In [ ]:
df['last_five'] = last_five

In [ ]:
last_five

In [ ]:
df.head()

In [ ]:
final_df = df.groupby('match_id').sum()['runs'].reset_index().merge(df, on='match_id')

In [ ]:
final_df

In [ ]:
final_df.columns

In [ ]:
final_df = final_df[['batting_team', 'bowling_team', 'city', 'current_score', 'balls_left', 'wicket_left',
       'current_run_rate', 'last_five', 'runs_x']]

In [ ]:
final_df.dropna(inplace=True)

In [ ]:
final_df

In [ ]:
final_df.isnull().sum()

In [ ]:
final_df.shape

In [ ]:
final_df = final_df.sample(final_df.shape[0])

In [ ]:
final_df.shape

In [ ]:
X = final_df.drop(columns=['runs_x'])
y = final_df['runs_x']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
transformer = ColumnTransformer([
    ('transformer', OneHotEncoder(sparse=False, drop='first'),['batting_team','bowling_team', 'city'])
], remainder='passthrough')

In [ ]:
pipe = Pipeline(steps=[
    ('step1', transformer),
    ('step2', StandardScaler()),
    ('step3', XGBRegressor(n_estimators=1000, learning_rate=0.2, max_depth=12, random_state=1))
])

In [ ]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [ ]:
r2_score(y_test, y_pred)

In [ ]:
mean_absolute_error(y_test, y_pred)

In [ ]:
import pickle
pickle.dump(pipe, open('pipe.pkl', 'wb'))